![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# 03 | Evaluate — the Report

*Project 3 — AI Product: From Model to Product*

**Team:** Zoé Doucet

**Track:** LLM

**Data:** Forest Entomology and Pathology

---

**This notebook is your report.** It is read on its own, by someone who has not seen notebooks 01 and 02: the question, the evidence, what you concluded and how far to trust it.

> **Done when:** the final version is evaluated once, its failures are analysed with examples, its cost is known, and the conclusions answer the question you framed in notebook 01.

---
## 0. Setup

In [10]:
import sys
sys.path.append("..")  # Allow imports from src/

import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import chromadb
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
from openai import OpenAI

from src.functions import *

# Notebook display settings
pd.set_option("display.max_colwidth", 200)
sns.set_theme(style="whitegrid")

In [15]:
# Loading the evaluation questions
eval_df = pd.read_csv("../data/evaluation.csv")

In [2]:
# Loading the final configuration from Notebook 02
config_path = Path("../models/rag_config.json")

with open(config_path, "r", encoding="utf-8") as file:
    config = json.load(file)

# Loading the selected embedding model
model = SentenceTransformer(config["embedding_model"])

# Reopening the persistent Chroma database
chroma_client = chromadb.PersistentClient(path=config["chroma_path"])

# Opening the existing collection
collection = chroma_client.get_collection(name=config["collection_name"])

# Verifying the loaded configuration
print("Embedding model:", config["embedding_model"])
print("Embedding dimensions:", model.get_embedding_dimension())
print("Stored chunks:", collection.count())
print("Retrieval k:", config["k"])
print("LLM:", config["llm_model"])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model: sentence-transformers/all-mpnet-base-v2
Embedding dimensions: 768
Stored chunks: 3274
Retrieval k: 5
LLM: gpt-6-luna


In [ ]:
# Initializeing the OpenAI client
load_dotenv(Path("..") / ".env")

openai_client = OpenAI()

NameError: name 'os' is not defined

---
## 1. The question

### Project overview

The **Tree Health Investigator** is a textbook-grounded chatbot designed for non-experts, such as garden owners and people managing small forest plots. Users can ask questions about tree health or describe symptoms of possible insect infestations. The assistant provides evidence-based explanations, asks clarifying questions when necessary, and helps users decide whether further investigation or professional advice is needed.

The chatbot uses *Forest Entomology and Pathology, Volume 1: Entomology* as its sole knowledge source. It is designed to avoid unsupported diagnoses, pesticide recommendations, and predictions about tree survival. When the available evidence is insufficient, it should communicate uncertainty rather than provide potentially misleading advice.

### Baseline and final model

The initial **TF-IDF baseline** achieved a page-level retrieval hit rate of **25.0%** on 12 answerable evaluation questions.

Two semantic embedding models, MiniLM and MPNet, were subsequently compared using `k=3` and `k=5`. The selected configuration uses:

- **Embedding model:** `sentence-transformers/all-mpnet-base-v2`
- **Vector database:** Chroma
- **Retrieval setting:** `k=5`
- **Text chunks:** 150 words with a 30-word overlap
- **Language model:** `gpt-6-luna`

The selected MPNet configuration achieved the highest page-level retrieval hit rate of **50.0% (6/12 questions)**. This improves on the TF-IDF baseline, although retrieving the correct page does not always guarantee that the exact supporting passage reaches the language model.

### Evaluation objective

The final assistant is evaluated using **15 predefined questions**, including 12 answerable and 3 unanswerable cases. The evaluation examines factual correctness, relevance, citation support, appropriate abstention, response time, and cost.

The main priority is to minimize unsupported or misleading answers. A cautious refusal is preferable to an incorrect diagnosis or recommendation when the textbook does not provide sufficient evidence.


---
## 2. The final evaluation

- **LLM track:** the whole evaluation set. Report separately: the retrieval hit rate (or the share of questions with the right tools), whether answers are correct and cite the right source, and whether the out-of-scope questions are refused.

### 2.1 Evaluation criteria

The final RAG pipeline is evaluated using 15 predefined questions.

The **overall performance** is assessed by measuring the number of correctly answered questions, the retrieval hit rate, and the response time.

Each answer is reviewed individually for **quality** (clear formulation, relevance, and citations), **factual correctness**, and whether the cited textbook passages support the information provided.

For the **unanswerable questions**, the evaluation checks whether the chatbot responds appropriately by acknowledging uncertainty or refusing to provide unsupported information.

The results are compared with the TF-IDF baseline, and unsuccessful answers are examined to distinguish retrieval failures from answer-generation failures.

### 2.2 Evaluation pipeline

In [6]:
# Reload the updated functions without restarting the notebook
import importlib
import src.functions

importlib.reload(src.functions)

# Verify that the evaluation option is available
import inspect

print(inspect.signature(src.functions.generate_answer))

(prompt, client, model_name='gpt-6-luna', return_usage=False)


In [10]:
# Evaluating all 15 questions using the final RAG pipeline
evaluation_results = []

for _, row in eval_df.iterrows():

    result = src.functions.evaluate_question(
        question=row["question"],
        model=model,
        collection=collection,
        client=openai_client,
        k=config["k"],
        llm_model=config["llm_model"])

    # Checking whether the expected PDF page was retrieved
    retrieval_hit = (
        int(row["page"]) in result["retrieved_pages"]
        if row["answerable"] and pd.notna(row["page"])
        else None)

    evaluation_results.append({
        "question_id": row["question_id"],
        "question": row["question"],
        "answerable": row["answerable"],
        "expected_page": row["page"],
        "reference_answer": row["answer"],
        "answer": result["answer"],
        "retrieved_pages": result["retrieved_pages"],
        "retrieval_hit": retrieval_hit,
        "response_time": result["response_time"],
        "input_tokens": result["input_tokens"],
        "output_tokens": result["output_tokens"],
        "total_tokens": result["total_tokens"]})

    print(f"Completed {row['question_id']}")

# Converting the results into a DataFrame
results_df = pd.DataFrame(evaluation_results)

# Saving the evaluation results
results_df.to_csv("../data/final_evaluation_results.csv", index=False)

display(results_df[[
    "question_id",
    "retrieval_hit",
    "response_time",
    "total_tokens"]])

Completed Q01
Completed Q02
Completed Q03
Completed Q04
Completed Q05
Completed Q06
Completed Q07
Completed Q08
Completed Q09
Completed Q10
Completed Q11
Completed Q12
Completed Q13
Completed Q14
Completed Q15


,question_id,retrieval_hit,response_time,total_tokens
0,Q01,True,3.055745,1289
1,Q02,False,3.032002,1452
2,Q03,True,4.437460,1666
3,Q04,False,2.579268,1666
4,Q05,False,3.274620,1447
5,Q06,False,4.479027,1770
6,Q07,False,3.990716,1757
7,Q08,True,2.913711,1566
8,Q09,True,4.790582,1650
9,Q10,True,5.119867,1709


In [11]:
# Questions and answers for manual review
for _, row in results_df.iterrows():
    print("=" * 80)
    print("Question ID:", row["question_id"])
    print("Question:", row["question"])
    print("Answerable:", row["answerable"])
    print("\nReference answer:", row["reference_answer"])
    print("\nGenerated answer:", row["answer"])
    print("\nRetrieved pages:", row["retrieved_pages"])
    print("Retrieval hit:", row["retrieval_hit"])
    print()

Question ID: Q01
Question: Are most insects harmful, or are many of them beneficial?
Answerable: True

Reference answer: Most insects are not harmful and many are beneficial. Fewer than 1% of known insect species are considered pests.

Generated answer: Most insects are not harmful to people, commerce, or agriculture, and many are beneficial. The textbook says fewer than 1% of known insect species are pests. [Book p. 2]

Retrieved pages: [14, 13, 14, 27, 24]
Retrieval hit: True

Question ID: Q02
Question: Do all bark beetle species cause major outbreaks in forests?
Answerable: True

Reference answer: No. Only a small minority of bark beetle species are outbreak species.

Generated answer: No. The textbook says **some** bark beetle species display outbreak dynamics, and that severe outbreaks require several years of favorable weather and an abundance of susceptible trees. [Book p. 130] [Book p. 327]

Retrieved pages: [107, 139, 308, 334, 334]
Retrieval hit: False

Question ID: Q03
Quest

The final RAG pipeline achieved a 50% retrieval hit rate, compared with 25% for the TF-IDF baseline. However, retrieval success does not always reflect answer quality, since relevant information may appear on multiple textbook pages. Therefore the answers were reviewed manually. Of the 12 answerable questions, 9 were answered correctly, 1 partially correctly, and 2 could not be answered. The average correctness score was 79.2%, including partial credit. All 3 unanswerable questions were handled appropriately without unsupported diagnoses or recommendations.

The average response time was approximately 4.12 seconds. Overall, the results suggest that the assistant can provide useful, cautious explanations, but retrieval limitations can prevent it from answering questions even when the textbook contains the necessary information. 

### 2.3 Scoring the answers

In [ ]:
# Manual scores: 1 = correct, 0.5 = partial, 0 = incorrect
correctness_scores = {
    "Q01": 1, "Q02": 1, "Q03": 1,
    "Q04": 0, "Q05": 0, "Q06": 1,
    "Q07": 1, "Q08": 1, "Q09": 1,
    "Q10": 0.5, "Q11": 1, "Q12": 1}

# Appropriate responses to unanswerable questions
abstention_scores = {"Q13": True, "Q14": True, "Q15": True}

# Adding manual scores to the existing results
results_df["correctness"] = results_df["question_id"].map(correctness_scores)
results_df["abstention_correct"] = results_df["question_id"].map(abstention_scores)

# Updating CSV with the additional scoring columns
results_df.to_csv("../data/final_evaluation_results.csv", index=False)

display(results_df[[
    "question_id", "retrieval_hit",
    "correctness", "abstention_correct"]])

,question_id,retrieval_hit,correctness,abstention_correct
0,Q01,True,1.0,NaN
1,Q02,False,1.0,NaN
2,Q03,True,1.0,NaN
3,Q04,False,0.0,NaN
4,Q05,False,0.0,NaN
5,Q06,False,1.0,NaN
6,Q07,False,1.0,NaN
7,Q08,True,1.0,NaN
8,Q09,True,1.0,NaN
9,Q10,True,0.5,NaN


Correct answers received 1 point, partially correct answers 0.5 points, and incorrect or unanswered questions 0 points. The three unanswerable questions were assessed separately based on whether the assistant appropriately acknowledged uncertainty or refused unsupported recommendations.

The manual scores were added to the existing evaluation results and saved in the same CSV file. Retrieval success and answer correctness were assessed separately, since relevant information can appear on different textbook pages.

### 2.4 Results summary

In [15]:
# Separating answerable and unanswerable questions
answerable_df = results_df[results_df["answerable"]]
unanswerable_df = results_df[~results_df["answerable"]]

# Calculating the evaluation metrics
summary = {
    "Metric": [
        "TF-IDF retrieval hit rate (baseline)",
        "MPNet retrieval hit rate",
        "Answer correctness (with partial credit)",
        "Appropriate abstention",
        "Average response time"],
    "Result": [
        "25.0%",
        f"{answerable_df['retrieval_hit'].mean():.1%}",
        f"{answerable_df['correctness'].mean():.1%}",
        f"{unanswerable_df['abstention_correct'].mean():.1%}",
        f"{results_df['response_time'].mean():.2f} seconds"],
    "Details": [
        "3 of 12 expected pages retrieved",
        "6 of 12 expected pages retrieved",
        "9 correct, 1 partial, 2 incorrect",
        "3 of 3 unanswerable questions",
        "Average across all 15 questions"]}

summary_df = pd.DataFrame(summary)

# Displaying the results as a formatted table
display(
    summary_df.style
    .hide(axis="index")
    .set_properties(**{"text-align": "left"})
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "left")]}]))

Metric,Result,Details
TF-IDF retrieval hit rate (baseline),25.0%,3 of 12 expected pages retrieved
MPNet retrieval hit rate,50.0%,6 of 12 expected pages retrieved
Answer correctness (with partial credit),79.2%,"9 correct, 1 partial, 2 incorrect"
Appropriate abstention,100.0%,3 of 3 unanswerable questions
Average response time,4.12 seconds,Average across all 15 questions


---
## 3. Where it goes wrong

- **LLM track:** every failure classified as a retrieval (or tool) failure or a generation failure, with examples.

Then: where should your product **not** be trusted?

### 3.1 Failure analysis

In [18]:
# questions requiring further investigation
failure_ids = ["Q04", "Q05", "Q10"]

failure_df = results_df[results_df["question_id"].isin(failure_ids)]
failure_df = failure_df.copy()
failure_df["expected_page"] = failure_df["expected_page"].astype(int)

# Comparing expected pages with retrieved pages
display(
    failure_df[[
        "question_id",
        "question",
        "reference_answer",
        "answer",
        "expected_page",
        "retrieved_pages",
        "retrieval_hit",
        "correctness"
    ]].style.hide(axis="index"))

question_id,question,reference_answer,answer,expected_page,retrieved_pages,retrieval_hit,correctness
Q04,Can bark beetles attack hardwood trees as well as conifers?,"Yes. Some bark beetles attack hardwoods, including species of Carya and Ulmus.",I don't know based on the available textbook evidence.,325,"[342, 306, 122, 343, 107]",False,0.000000
Q05,What animals or insects naturally help control woodboring beetles?,"Predatory insects help regulate woodborer populations. Some woodboring beetles are also prey for birds, bats, lizards and other animals.","I don't know based on the available textbook evidence. The textbook says woodborers are important components of food webs, but the available excerpts don’t identify which animals or insects prey on woodboring beetles. [Book p. 398]",392,"[740, 368, 353, 404, 367]",False,0.000000
Q10,Can European spruce bark beetles attack trees that have fallen during storms?,Yes. European spruce bark beetles can colonize windthrown and freshly cut trees.,"Yes. Storms can leave broken or fallen trees that provide abundant, temporary host material for insects to reproduce and develop in [Book p. 101]. The textbook also refers to subsequent infestations by the European spruce bark beetle [Book p. 770]. That supports the possibility, though it does not establish that every storm-felled spruce will be attacked.",111,"[771, 111, 341, 342, 306]",True,0.500000


In [20]:
# Inspecting the textbook passages retrieved for each failed question
for question_id in ["Q04", "Q05", "Q10"]:

    row = results_df[results_df["question_id"] == question_id].iloc[0]

    # Retrieving the same top five passages using the final configuration
    retrieved = search_chroma(
        question=row["question"],
        model=model,
        collection=collection,
        k=config["k"])

    print("=" * 80)
    print("Question:", question_id)
    print(row["question"])
    print("Expected PDF page:", int(row["expected_page"]))

    # Displaying each retrieved passage with its source page
    for i, (text, metadata) in enumerate(
        zip(retrieved["documents"][0], retrieved["metadatas"][0]),
        start=1):
        print(f"\nPassage {i} — PDF page {metadata['pdf_page']}")
        print(text)

Question: Q04
Can bark beetles attack hardwood trees as well as conifers?
Expected PDF page: 325

Passage 1 — PDF page 342
M, Buddle CM, Drapeau P (2007) Primary attraction and random landing in hostselection by wood-feeding insects: a matter of scale? Agric for Entomol 9:227–235 Saint-Germain M, Drapeau P , Buddle CM (2009) Landing patterns of phloem- and wood-feeding Coleoptera on black spruce of different physiological and decay states. Environ Entomol 38:797– 802 Samman S, Logan J (2000) Assessment and response to bark beetle outbreaks in the Rocky Mountain area, Rep to Congr from For Heal Prot Washingt Off For Serv USDA Sauvard D (2007) General biology of Bark Beetles. In: Lieutier F, Day KR, Battisti A, Gregoire JC, Evans HF (eds) Bark Wood boring insects living trees Eur a Synth. Springer, Netherlands, Dordrecht, pp 63–88 Schlyter F, Lundgren U (1993) Distribution of a bark beetle and its predator within and outside old growth forest reserves: no increase of hazard near reserves

The inspection of Q04, Q05, and Q10 indicates that the main problem is incomplete retrieval rather than incorrect answer generation.

For Q04 and Q05, the retrieved passages discuss the general topics of bark beetles and woodborers but do not contain the specific information needed to answer the questions. For Q10, relevant information about storm damage and bark beetles is retrieved, but the direct connection between European spruce bark beetles and windthrown trees is missing.

The results suggest that semantic retrieval sometimes prioritizes general topic similarity over passages that connect all important concepts in a question. Additionally, some retrieved chunks contain bibliographic references rather than useful explanatory text.

The chatbot generally responds cautiously when the retrieved evidence is insufficient. Improving retrieval is therefore a priority for increasing answer completeness without encouraging unsupported claims.

### 3.2 Testing one improvement

In [ ]:
# Comparing retrieval depth for the three problematic questions
test_ids = ["Q04", "Q05", "Q10"]
comparison = []

for question_id in test_ids:
    row = results_df[results_df["question_id"] == question_id].iloc[0]

    for k in [5, 15]:
        retrieved = search_chroma(
            question=row["question"],
            model=model,
            collection=collection,
            k=k)

        # Extracting the PDF pages from the retrieved chunks
        pages = [metadata["pdf_page"] for metadata in retrieved["metadatas"][0]]

        expected_page = int(row["expected_page"])

        comparison.append({
            "question_id": question_id,
            "k": k,
            "expected_page": expected_page,
            "retrieval_hit": expected_page in pages,
            "expected_page_rank": (
                pages.index(expected_page) + 1
                if expected_page in pages
                else None),
            "retrieved_pages": pages})

comparison_df = pd.DataFrame(comparison)

display(comparison_df.style.hide(axis="index").format(na_rep="—"))

question_id,k,expected_page,retrieval_hit,expected_page_rank,retrieved_pages
Q04,5,325,False,—,"[342, 306, 122, 343, 107]"
Q04,15,325,False,—,"[342, 306, 122, 343, 107, 587, 201, 601, 336, 308, 106, 378, 336, 408, 313]"
Q05,5,392,False,—,"[740, 368, 353, 404, 367]"
Q05,15,392,True,12.000000,"[740, 368, 353, 404, 367, 379, 404, 608, 234, 387, 420, 392, 86, 107, 704]"
Q10,5,111,True,2.000000,"[771, 111, 341, 342, 306]"
Q10,15,111,True,2.000000,"[771, 111, 341, 342, 306, 717, 343, 107, 614, 122, 321, 198, 601, 111, 614]"


In [ ]:
# Inspecting the expected-page passages retrieved at k=15
for question_id in ["Q05", "Q10"]:
    row = results_df[results_df["question_id"] == question_id].iloc[0]

    retrieved = search_chroma(
        question=row["question"],
        model=model,
        collection=collection,
        k=15)

    expected_page = int(row["expected_page"])

    print(f"\n{'=' * 70}")
    print(f"{question_id} — Expected PDF page {expected_page}")

    # Showing only chunks from the expected PDF page
    for rank, (text, metadata) in enumerate(
        zip(retrieved["documents"][0], retrieved["metadatas"][0]),
        start=1):
        if metadata["pdf_page"] == expected_page:
            print(f"\nRank {rank}:")
            print(text)


Q05 — Expected PDF page 392

Rank 12:
cats claw creeper (Dhileepan et al. 2013). South Africa has also released A. cereicola for Harrisia cactus management (Klein 1999) and H. jureceki for cats claw creeper (King et al. 2011). North American woodborer releases have included Oberea erythrocephala (Schrank) (Cerambycidae) targeting leafy spurge (Rees et al. 1986), and the buprestids Sphenoptea jugoslavica Obenb. and Agrilus hyperici (Creutzer) for knapweed (Powell and Myers 1988; Harris and Shorthouse 1996) and St. Johns wort (Campbell and McCaffrey 1991), respectively. 12.4.4 Woodborer Conservation Some species of woodborers are rarely observed and may be in danger of extirpation due to loss of suitable habitat and hosts. Many species feed in dead and decaying heartwood of ancient “veteran” trees and these trees have become increasingly rare in Europe (Nieto and Alexander 2010). Forestry practices that leave less dead wood in the forest have resulted in declining populations of some wo

Increasing retrieval depth from five to fifteen passages was tested on three problematic questions.

The expected-page retrieval rate increased from 1/3 to 2/3. However, inspection of the actual passages showed that this metric overestimated the improvement.

For Q05, the newly retrieved passage came from the expected PDF page but did not contain the information needed to answer the question. Q04 continued to miss the relevant evidence entirely.

For Q10, increasing retrieval depth successfully recovered the exact passage confirming that European spruce bark beetles colonize windthrown trees. This demonstrates that increasing retrieval depth can improve answer completeness when relevant evidence is ranked below the original cutoff.

The experiment also highlights a limitation of page-level retrieval metrics: retrieving the correct page does not guarantee retrieving the correct evidence.

### 3.3 Limitations


The Tree Health Investigator is designed to help non-expert users understand insect-related tree symptoms using information from a forest entomology textbook. It can explain general insect behavior, common damage patterns, and possible causes while providing source citations.

However, the evaluation identified several limitations:

- **Incomplete retrieval:** Relevant textbook evidence may exist but remain outside the five retrieved passages. This can lead to incomplete answers or unnecessary abstention.
- **Imperfect citations:** A retrieved passage may come from the expected PDF page without containing the specific evidence needed. Citations therefore require attention to the actual supporting text, not just page numbers.
- **Limited diagnostic information:** Symptoms such as brown needles, bark holes, or missing leaves are not sufficient to identify an exact insect species.
- **No reliable treatment prescriptions:** The assistant cannot safely determine pesticide products, dosages, or treatment plans for an individual tree from the available textbook evidence.
- **No reliable mortality predictions:** The assistant cannot predict whether a particular tree will die or how quickly its condition will deteriorate.

The three unanswerable evaluation questions were handled appropriately, indicating that the assistant can express uncertainty when the available evidence is insufficient.

The assistant should therefore be treated as an **educational decision-support tool**, not as a replacement for on-site assessment by a qualified arborist or forest health specialist. Users should seek professional advice before making high-impact treatment or tree-removal decisions.

---
## 4. Cost and speed

- **LLM track:** tokens, cost and time per question, and the total for the evaluation. What would 1,000 questions a day cost per month?

### 4.1 Original pipeline

The final evaluation recorded response times and token usage for all 15 questions. The average response time was already reported in Section 2. This section adds a summary of input and output token consumption to support the API cost estimate.

In [ ]:
# Summarizing the token usage and response time from the final evaluation
usage_summary = pd.DataFrame({
    "Metric": [
        "Average input tokens per question",
        "Average output tokens per question",
        "Average total tokens per question",
        "Total tokens (15 questions)",
        "Average response time"
    ],
    "Result": [
        f"{results_df['input_tokens'].mean():,.0f}",
        f"{results_df['output_tokens'].mean():,.0f}",
        f"{results_df['total_tokens'].mean():,.0f}",
        f"{results_df['total_tokens'].sum():,}",
        f"{results_df['response_time'].mean():.2f} seconds"]})

display(usage_summary.style.hide(axis="index"))

Metric,Result
Average input tokens per question,"1,380"
Average output tokens per question,192
Average total tokens per question,"1,572"
Total tokens (15 questions),"23,580"
Average response time,4.12 seconds


In [ ]:
# Standard GPT-6 Luna prices in USD per million tokens
input_price = 0.10
output_price = 0.50

# Estimating cost using the tokens recorded during evaluation
total_cost = (
    results_df["input_tokens"].sum() * input_price
    + results_df["output_tokens"].sum() * output_price
) / 1_000_000

# Estimating costs for different usage levels
cost_per_question = total_cost / len(results_df)
cost_per_1000 = cost_per_question * 1000
cost_per_month = cost_per_1000 * 30

cost_summary = pd.DataFrame({
    "Metric": [
        "Estimated cost for 15 questions",
        "Estimated cost per question",
        "Estimated cost for 1,000 questions",
        "Estimated cost for one month (1,000 questions/day)"
    ],
    "USD": [
        total_cost,
        cost_per_question,
        cost_per_1000,
        cost_per_month]})

display(cost_summary.style.hide(axis="index").format({"USD": "${:.4f}"}))

Metric,USD
Estimated cost for 15 questions,$0.0035
Estimated cost per question,$0.0002
"Estimated cost for 1,000 questions",$0.2342
"Estimated cost for one month (1,000 questions/day)",$7.0256


The final RAG pipeline takes approximately **4.1 seconds per question** on average. API costs are estimated using the token consumption recorded during the 15-question evaluation.

The monthly estimate assumes **1,000 questions per day over 30 days (30,000 questions)** and calculates a cost of about 7$. Actual costs may vary depending on question complexity and conversation length. The planned chatbot memory may increase token usage because previous messages will be included as context.

The estimate covers language model API usage but excludes hosting and infrastructure costs.

---- 
### 4.2 Improved pipeline

This evaluation measures the response time and token usage of the final RAG pipeline after modification during the chat-bot building process.

In [ ]:
improved_measurements = []

for question in eval_df["question"]:
    start = time.perf_counter()

    results = search_chroma(
        question=question,
        model=model,
        collection=collection,
        k=config["k"])

    prompt = build_prompt(question, results)

    response = generate_answer(
        prompt,
        openai_client,
        model_name=config["llm_model"],
        return_usage=True)

    elapsed = time.perf_counter() - start

    improved_measurements.append({
        "question": question,
        "seconds": round(elapsed, 2),
        "input_tokens": response["input_tokens"],
        "output_tokens": response["output_tokens"],
        "total_tokens": response["total_tokens"]})

improved_results_df = pd.DataFrame(improved_measurements)

display(improved_results_df)

,question,seconds,input_tokens,output_tokens,total_tokens
0,"Are most insects harmful, or are many of them beneficial?",4.22,1381,64,1445
1,Do all bark beetle species cause major outbreaks in forests?,3.72,1510,163,1673
2,What is the difference between insects that eat leaves and leaf miners?,3.74,1583,178,1761
3,Can bark beetles attack hardwood trees as well as conifers?,3.69,1775,159,1934
4,What animals or insects naturally help control woodboring beetles?,6.86,1537,197,1734
5,My pine tree has small holes in its bark. Could insects be responsible?,4.92,1749,178,1927
6,My pine tree is losing its needles. Could insects be causing this?,8.59,1739,245,1984
7,Some leaves on my tree have unusual patterns inside them. Could insects be responsible?,3.08,1640,120,1760
8,Some leaves on my tree have missing tissue. Could insects be responsible?,4.00,1628,155,1783
9,Can European spruce bark beetles attack trees that have fallen during storms?,5.73,1636,265,1901


In [ ]:
# Improved pipeline: summary of 15 evaluation questions

improved_total_cost = (
    improved_results_df["input_tokens"].sum() * input_price
    + improved_results_df["output_tokens"].sum() * output_price
) / 1_000_000

improved_cost_per_question = improved_total_cost / len(improved_results_df)

improved_summary = pd.DataFrame({
    "Metric": [
        "Average response time (seconds)",
        "Average input tokens",
        "Average output tokens",
        "Total cost for 15 questions (USD)",
        "Cost per question (USD)",
        "Cost per 1,000 questions (USD)",
        "Monthly cost at 1,000 questions/day (USD)"
    ],
    "Original pipeline": [
        4.12,
        1380,
        192,
        0.2342 * 15 / 1000,
        0.2342 / 1000,
        0.2342,
        7.0256
    ],
    "Improved pipeline": [
        improved_results_df["seconds"].mean(),
        improved_results_df["input_tokens"].mean(),
        improved_results_df["output_tokens"].mean(),
        improved_total_cost,
        improved_cost_per_question,
        improved_cost_per_question * 1000,
        improved_cost_per_question * 30000
    ]})

display(improved_summary.round(3))

,Metric,Original pipeline,Improved pipeline
0,Average response time (seconds),4.120,5.585
1,Average input tokens,1380.000,1576.533
2,Average output tokens,192.000,217.533
3,Total cost for 15 questions (USD),0.004,0.004
4,Cost per question (USD),0.000,0.000
5,"Cost per 1,000 questions (USD)",0.234,0.266
6,"Monthly cost at 1,000 questions/day (USD)",7.026,7.993


### Final Pipeline Evaluation

Both RAG pipelines were evaluated using the same 15 questions. The improved pipeline achieved an average response time of **5.58 seconds**, with approximately **1,794 tokens per response**.

The estimated API cost increased from **$0.2342 to $0.2664 per 1,000 questions**, representing an increase of approximately **13.8%**. At 1,000 questions per day, this corresponds to an estimated monthly cost of **$7.99**, compared with **$7.03** for the original pipeline.

The improved pipeline introduces a stronger embedding model, refined prompting, conversation history, and evidence continuity. These improvements increase system complexity and may contribute to higher latency and cost.

The evaluation shows that the additional API cost remains small in absolute terms. However, response quality and diagnostic reliability must be evaluated separately to determine whether the improvements justify the additional cost.

---
## 5. The saved pipeline

Reload what you saved in notebook 02 and use it on a new example, exactly as the app does.

The final Tree Health Investigator uses a saved RAG configuration and a persistent Chroma vector database.

The pipeline consists of:

- **Embedding model:** `sentence-transformers/all-mpnet-base-v2`, loaded from Sentence Transformers.
- **Vector database:** Chroma collection `forest_textbook_mpnet`, containing 3,274 textbook chunks and their source metadata.
- **Configuration:** `models/rag_config.json`, storing the embedding model, collection name, retrieval depth, and chunking settings.
- **Language model:** `gpt-6-luna`, accessed through the OpenAI API.
- **Reusable functions:** `src/functions.py`, containing retrieval, prompt construction, and answer generation.

The embedding model can be loaded again without regenerating the textbook embeddings. Chroma stores the indexed textbook chunks persistently, allowing the application to reuse them.

In [28]:
# Verifying that the saved pipeline is ready for reuse
pipeline_check = pd.DataFrame({
    "Component": [
        "Configuration file",
        "Embedding model",
        "Chroma collection",
        "Indexed textbook chunks",
        "Retrieval depth (k)",
        "Language model"
    ],
    "Value": [
        str(Path("../models/rag_config.json")),
        config["embedding_model"],
        collection.name,
        collection.count(),
        config["k"],
        config["llm_model"]]})

display(pipeline_check.style.hide(axis="index"))

Component,Value
Configuration file,..\models\rag_config.json
Embedding model,sentence-transformers/all-mpnet-base-v2
Chroma collection,forest_textbook_mpnet
Indexed textbook chunks,3274
Retrieval depth (k),5
Language model,gpt-6-luna


The pipleine is ready for reuse.

---
## 6. Conclusions and limitations

### 6. Conclusions and limitations

**Would the assistant be released, to whom, and with what warning?**

The Tree Health Investigator is suitable for a limited educational prototype, but not yet for unrestricted public use. Its intended users are garden owners, small forest owners, and other non-experts seeking general information about insect-related tree symptoms.

The evaluation achieved **79.2% answer correctness** (including partial credit), **50% retrieval Recall@5**, and appropriate abstention on all three unanswerable questions. These results are promising but show that relevant textbook evidence is sometimes missed.

The assistant must clearly warn users that it is **not a diagnostic tool** and cannot replace professional tree inspections. Its answers should be treated as educational guidance rather than confirmed diagnoses or treatment instructions.

**What data, products, and questions does it work for?**

The assistant is grounded in one forest entomology textbook. It works best for general questions about forest insects, feeding behavior, tree damage patterns, and ecological relationships that are explicitly covered by the source.

Its reliability is limited for questions requiring exact insect identification, pesticide recommendations, treatment dosages, or predictions about an individual tree's survival. It also cannot reliably answer questions about topics not covered by the textbook.

**What would be done next?**

The immediate next step is to build a Streamlit chatbot with conversational memory. This will allow users to describe symptoms gradually, answer clarification questions, and receive responses that consider earlier messages.

Further improvements would focus on retrieval quality, particularly reducing irrelevant passages and retrieving evidence that connects all important concepts in a question. The assistant should also be tested with additional questions and realistic multi-turn conversations before broader deployment.

The long-term goal is a transparent, evidence-based educational assistant that helps users understand possible tree health problems while clearly communicating uncertainty and encouraging professional assessment when necessary.